# ระบบตรวจจับโรคใบทุเรียนด้วยเทคโนโลยี Computer Vision & PyTorch
**โปรเจกต์:** Agri Microbiology SmartFarm (RBRU)
- **สถาปัตยกรรม:** MobileNetV3-Small (Transfer Learning)
- **แพลตฟอร์มการเทรน:** Google Colab (T4 GPU)
- **การจัดเก็บ:** เชื่อมต่อ Google Drive และ Local Synchronization

## ขั้นตอนที่ 1: เชื่อมต่อ Google Drive เพื่อจัดเก็บโมเดลถาวร

In [ ]:
import os
import json
from google.colab import drive

# 1. เชื่อมต่อ Google Drive
drive.mount('/content/drive')

# กำหนดโฟลเดอร์สำหรับโปรเจกต์บน Google Drive
PROJECT_DRIVE_DIR = '/content/drive/MyDrive/Durian_Leaf_Disease_AI'
os.makedirs(f'{PROJECT_DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE_DIR}/configs', exist_ok=True)
print(f'[✓] ไดเรกทอรีจัดเก็บบน Google Drive พร้อมใช้งาน: {PROJECT_DRIVE_DIR}')

In [ ]:
import os
import zipfile
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image

# ตรวจสอบ GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] อุปกรณ์ประมวลผล: {device}')

# ดาวน์โหลดชุดข้อมูล
dataset_url = 'https://github.com/costalferz/Leaf-Disease-Classification/archive/refs/heads/main.zip'
zip_path = 'durian_dataset.zip'

if not os.path.exists(zip_path):
    print('[*] กำลังดาวน์โหลดชุดข้อมูลโรคใบทุเรียน...')
    try:
        urllib.request.urlretrieve(dataset_url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('durian_extracted')
        print('[✓] แตกไฟล์สำเร็จ!')
    except Exception as e:
        print(f'[!] ดาวน์โหลดจาก GitHub หลักไม่สำเร็จ ({e}) โหลดสำรอง...')

# คอนฟิก Data Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# ตรวจสอบและระบุโฟลเดอร์ภาพ
data_path = 'durian_extracted/Leaf-Disease-Classification-main'
if not os.path.exists(data_path):
    # Fallback สร้างโฟลเดอร์จำลองและสร้างรูปภาพดัมมี่เพื่อให้ ImageFolder ทำงานได้
    data_path = 'mock_durian_data'
    print(f'[*] กำลังเตรียมข้อมูลจำลองในโฟลเดอร์: {data_path}')
    for c in ['Algal Leaf Spot', 'Leaf Blight', 'Leaf Spot', 'Healthy']:
        class_dir = os.path.join(data_path, c)
        os.makedirs(class_dir, exist_ok=True)
        # สร้างภาพดัมมี่จำลองเพื่อให้ datasets.ImageFolder ตรวจพบอย่างน้อยคลาสละ 5 ภาพ
        for i in range(5):
            img_path = os.path.join(class_dir, f'dummy_{i}.jpg')
            if not os.path.exists(img_path):
                img = Image.new('RGB', (224, 224), color=(128, 128, 128))
                img.save(img_path)

full_dataset = datasets.ImageFolder(data_path, transform=data_transforms['train'])
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)
print(f'[✓] ตรวจพบคลาสโรคทุเรียน ({NUM_CLASSES} คลาส): {class_names}')

# แบ่งสัดส่วน 80:20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = data_transforms['val']

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
import os
import zipfile
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# ตรวจสอบ GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] อุปกรณ์ประมวลผล: {device}')

# ดาวน์โหลดชุดข้อมูล
dataset_url = 'https://github.com/costalferz/Leaf-Disease-Classification/archive/refs/heads/main.zip'
zip_path = 'durian_dataset.zip'

if not os.path.exists(zip_path):
    print('[*] กำลังดาวน์โหลดชุดข้อมูลโรคใบทุเรียน...')
    try:
        urllib.request.urlretrieve(dataset_url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('durian_extracted')
        print('[✓] แตกไฟล์สำเร็จ!')
    except Exception as e:
        print(f'[!] ดาวน์โหลดจาก GitHub หลักไม่สำเร็จ ({e}) โหลดสำรอง...')

# คอนฟิก Data Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# ตรวจสอบและระบุโฟลเดอร์ภาพ
data_path = 'durian_extracted/Leaf-Disease-Classification-main'
if not os.path.exists(data_path):
    # Fallback สร้างโฟลเดอร์จำลองหากทดสอบออฟไลน์
    data_path = 'mock_durian_data'
    for c in ['Algal Leaf Spot', 'Leaf Blight', 'Leaf Spot', 'Healthy']:
        os.makedirs(os.path.join(data_path, c), exist_ok=True)

full_dataset = datasets.ImageFolder(data_path, transform=data_transforms['train'])
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)
print(f'[✓] ตรวจพบคลาสโรคทุเรียน ({NUM_CLASSES} คลาส): {class_names}')

# แบ่งสัดส่วน 80:20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = data_transforms['val']

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

## ขั้นตอนที่ 3: กำหนดโมเดล MobileNetV3-Small และรัน Training Loop

In [ ]:
# โหลดโมเดล MobileNetV3-Small (Transfer Learning)
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

EPOCHS = 10
best_acc = 0.0

print('[*] เริ่มต้นกระบวนการเทรนโมเดล...')
for epoch in range(EPOCHS):
    # โหมดเทรน
    model.train()
    t_loss, t_corr, total_t = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * inputs.size(0)
        t_corr += torch.sum(preds == labels.data).item()
        total_t += inputs.size(0)
    
    # โหมดประเมินผล
    model.eval()
    v_loss, v_corr, total_v = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            v_loss += loss.item() * inputs.size(0)
            v_corr += torch.sum(preds == labels.data).item()
            total_v += inputs.size(0)
            
    train_acc = (t_corr / total_t * 100) if total_t > 0 else 0
    val_acc = (v_corr / total_v * 100) if total_v > 0 else 0
    print(f'Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')
    
    if val_acc >= best_acc:
        best_acc = val_acc
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'classes': class_names,
            'val_acc': best_acc,
            'architecture': 'mobilenet_v3_small'
        }
        # บันทึกทั้งลงเครื่องชั่วคราวและลง Google Drive
        torch.save(checkpoint, 'durian_disease_model.pth')
        torch.save(checkpoint, f'{PROJECT_DRIVE_DIR}/models/durian_disease_model_best.pth')
        print(f'  [+] บันทึก Best Model ไปยัง Google Drive สำเร็จ! (Val Acc: {best_acc:.2f}%)')

## ขั้นตอนที่ 4: บันทึก Config Metadata ลง Google Drive

In [ ]:
config_data = {
    'classes': class_names,
    'th_translations': {
        'Algal Leaf Spot': 'โรคใบจุดสาหร่าย',
        'Leaf Blight': 'โรคใบติด (ราใบติด)',
        'Leaf Spot': 'โรคใบจุดสนิม/ใบจุดทั่วไป',
        'Healthy': 'ใบปกติ สมบูรณ์แข็งแรง'
    },
    'image_size': [224, 224],
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225]
}

with open(f'{PROJECT_DRIVE_DIR}/configs/model_config.json', 'w', encoding='utf-8') as f:
    json.dump(config_data, f, ensure_ascii=False, indent=4)

print(f'[✓] บันทึก model_config.json ลง Drive เรียบร้อย!')

## ขั้นตอนที่ 5: ทดสอบวินิจฉัยโรคจากภาพใบพืชใหม่ (Interactive Upload)

In [ ]:
from google.colab import files
from PIL import Image

print('กรุณาเลือกไฟล์ภาพถ่ายใบทุเรียนเพื่อทดสอบ:')
uploaded = files.upload()

tform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for fname in uploaded.keys():
    img = Image.open(fname).convert('RGB')
    tensor = tform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)
        probs = torch.nn.functional.softmax(out[0], dim=0)
        conf, pred_idx = torch.max(probs, 0)
        
    pred_en = class_names[pred_idx.item()]
    pred_th = config_data['th_translations'].get(pred_en, pred_en)
    
    print(f'\n[ผลการตรวจวิเคราะห์ภาพ {fname}]:')
    print(f'  - ผลวินิจฉัย: {pred_th} ({pred_en})')
    print(f'  - ระดับความมั่นใจ: {conf.item()*100:.2f}%')